# Euler–Bernoulli cantilever: accurate nonperiodic JAX evolution

Solve $\rho A w_{tt}+EI w_{xxxx}=q$ on $[0,1]$, initially at rest, under a
uniform load switched on at $t=0$. The left end is clamped ($w=w_x=0$);
the right end is free ($w_{xx}=w_{xxx}=0$). The model and forcing are unchanged.
All four conditions are imposed using full BSPF derivative rows.

Install `python -m pip install -e './jax[notebook]'` and use an x64 kernel.
For the affected OpenMP BLAS installation, start with `OMP_NUM_THREADS=1`
or use the corrected-BLAS launcher. First calls include compilation.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import bspf_jax as bspf
from bspf_jax.references import cantilever_spectrum, cantilever_step_response

## Resolved bending energy, consistent load, exact modal phases

`galerkin_1d(..., quadrature_order=8)` integrates the actual BSPF trial functions
and curvature between grid points. The load uses the same quadrature.
`integrate_elastic` computes frequencies from an SVD of the mass-scaled
**curvature factor**, avoiding the squared conditioning of the stiffness matrix.
It then uses exact sine/cosine phases for this constant-load linear system.
There is no time-step truncation error.

The trial space enforces the clamp and free-end conditions, not periodicity.
Independent derivative evaluations check all four residuals. High derivative
residuals amplify floating-point error and should not be confused with field
error. Strongly enforcing the free conditions slightly reduces the trial space;
spatial refinement checks that this still approximates the continuum solution.

In [ ]:
rhoA, EI, load = 1., 1., 1.
times = jnp.linspace(0., 3., 301)

def solve(n, quadrature_order=8):
    x = jnp.linspace(0., 1., n)
    spatial = bspf.plan_1d(x, degree=5, n_basis=16, boundary_points=7)
    weak = bspf.galerkin_1d(spatial, derivative_order=2,
        constraints=((0, 0), (0, 1), (1, 2), (1, 3)), quadrature_order=quadrature_order)
    force = load*(weak.values.T@weak.quadrature_weights)
    initial = jnp.zeros(weak.mass.shape[0])
    q, velocity = jax.jit(lambda: bspf.integrate_elastic(
        weak, initial, initial, times, force=force, density=rhoA, rigidity=EI))()
    return x, spatial, weak, force, q, velocity

x, spatial, weak, force, q, velocity = solve(129)
coarse_x, _, coarse_weak, _, coarse_q, _ = solve(65)
_, _, quadrature_weak, _, quadrature_q, _ = solve(129, quadrature_order=10)
displacement = q@weak.extension.T
coarse = coarse_q@coarse_weak.extension.T
quadrature_check = quadrature_q@quadrature_weak.extension.T

## Independent reference and complete accuracy checks

The continuum reference uses 256 analytic cantilever modes and checks against
512. Its roots solve $\cos b+\operatorname{sech}b=0$. Stable decaying-exponential
mode formulas avoid hyperbolic cancellation, and modal load coefficients are
integrated analytically. No BSPF matrix enters the reference.

Validate the static quartic solution, fundamental frequency, full transient
field, spatial/quadrature/reference refinement, and actual boundary residuals.
Compute energy about static equilibrium directly from quadrature-point curvature
and velocity. Forming a quadratic expression with the ill-conditioned stiffness
matrix would unnecessarily lose digits in this diagnostic.

In [ ]:
exact = cantilever_step_response(x, times, mode_count=256, density=rhoA, rigidity=EI, load=load)
reference_check = cantilever_step_response(x, times, mode_count=512, density=rhoA, rigidity=EI, load=load)
field_error = jnp.max(jnp.abs(displacement-exact), axis=1)
coarse_error = jnp.max(jnp.abs(coarse-cantilever_step_response(coarse_x, times)))
quadrature_error = jnp.max(jnp.abs(displacement-quadrature_check))
reference_error = jnp.max(jnp.abs(exact-reference_check))
frequencies, modes = bspf.elastic_modes(weak, density=rhoA, rigidity=EI)
static_q = modes@((modes.T@force)/frequencies**2)
static = weak.extension@static_q
static_exact = load*x**2*(x**2-4*x+6)/(24*EI)
static_error = jnp.max(jnp.abs(static-static_exact))
root, _ = cantilever_spectrum(1)
frequency_error = jnp.abs(frequencies[0]/(root[0]**2*jnp.sqrt(EI/rhoA))-1)
curvature = (q-static_q)@weak.derivative_values.T
quadrature_velocity = velocity@weak.values.T
energy = .5*(EI*curvature**2+rhoA*quadrature_velocity**2)@weak.quadrature_weights
energy_drift = jnp.abs(energy/energy[0]-1)
derivative = bspf.derivatives(spatial, displacement.T, orders=(1, 2, 3))
clamp_value = jnp.max(jnp.abs(displacement[:, 0]))
clamp_slope = jnp.max(jnp.abs(derivative[1][0]))
free_moment = jnp.max(jnp.abs(EI*derivative[2][-1]))
free_shear = jnp.max(jnp.abs(EI*derivative[3][-1]))
print(f"field error: {field_error.max():.3e}; coarse grid: {coarse_error:.3e}")
print(f"quadrature difference: {quadrature_error:.3e}; reference tail: {reference_error:.3e}")
print(f"static error: {static_error:.3e}; fundamental frequency relative error: {frequency_error:.3e}")
print(f"relative energy drift: {energy_drift.max():.3e}")
print(f"clamp value/slope: {clamp_value:.3e}, {clamp_slope:.3e}; free moment/shear: {free_moment:.3e}, {free_shear:.3e}")
assert jnp.all(jnp.isfinite(displacement))
assert field_error.max() < 5e-9
assert field_error.max() < coarse_error/5
assert quadrature_error < 1e-9
assert reference_error < 1e-12
assert static_error < 1e-10
assert frequency_error < 1e-9
assert energy_drift.max() < 1e-9
assert clamp_value < 1e-13 and clamp_slope < 1e-9
assert free_moment < 1e-8 and free_shear < 2e-6

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].plot(times, displacement[:, -1], label="JAX BSPF")
axes[0, 0].plot(times, exact[:, -1], "k--", label="256-mode reference")
axes[0, 0].set(xlabel="t", ylabel="Tip displacement", title="Cantilever response")
axes[0, 0].legend()
axes[0, 1].semilogy(times[1:], field_error[1:], label="129 samples")
coarse_reference = cantilever_step_response(coarse_x, times)
axes[0, 1].semilogy(times[1:], jnp.max(jnp.abs(coarse-coarse_reference), axis=1)[1:], label="65 samples")
axes[0, 1].set(xlabel="t", ylabel="Max displacement error", title="Spatial refinement")
axes[0, 1].legend()
im = axes[1, 0].pcolormesh(x, times, displacement-exact, shading="auto")
axes[1, 0].set(xlabel="x", ylabel="t", title="Signed field error")
fig.colorbar(im, ax=axes[1, 0])
axes[1, 1].plot(times, energy/energy[0]-1)
axes[1, 1].set(xlabel="t", ylabel="Relative energy drift", title="Energy about static equilibrium")
plt.show()